# Step 0 — 資料健檢

確認資料完整、欄位一致、無異常缺漏。
此步驟無需調整參數。

In [1]:
# ===== 共用參數 =====
exec(open('../config/params.py').read())
# ==============================

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml


In [3]:
df = pd.read_csv(DATA_PATH)
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format=DATETIME_FORMAT)

print("=" * 60)
print("資料健檢報告")
print("=" * 60)

# 1. 基本資訊
print(f"\n✓ 總筆數: {len(df):,}")
print(f"✓ 欄位數: {len(df.columns)}")
print(f"✓ 欄位: {list(df.columns)}")

# 2. Null 檢查
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)
null_report = pd.DataFrame({'null_count': nulls, 'null_pct': nulls_pct})
null_report = null_report[null_report['null_count'] > 0]
if len(null_report) > 0:
    print(f"\n⚠️  有 null 的欄位:")
    for col, row in null_report.iterrows():
        print(f"   {col}: {row['null_count']:,} ({row['null_pct']}%)")
else:
    print(f"\n✓ 無 null 欄位")

# 3. order_id 唯一性
dup = df['order_id'].duplicated().sum()
print(f"\n{'✓' if dup == 0 else '✗'} order_id 重複: {dup}")

# 4. Device 數量
print(f"✓ Unique devices: {df['device_id'].nunique()}")

# 5. 時間範圍
print(f"✓ 時間範圍: {df['order_created_at'].min()} ~ {df['order_created_at'].max()}")
print(f"  天數: {(df['order_created_at'].max() - df['order_created_at'].min()).days}")

# 6. 異常值檢查
checks = {
    'total_duration_seconds <= 0': (df['total_duration_seconds'] <= 0).sum(),
    'file_count <= 0': (df['file_count'] <= 0).sum(),
}
for col in df.select_dtypes(include='number').columns:
    if 'duration' in col or 'seconds' in col or 'minutes' in col:
        neg = (df[col] < 0).sum()
        if neg > 0:
            checks[f'{col} < 0'] = neg

for desc, count in checks.items():
    print(f"{'✓' if count == 0 else '✗'} {desc}: {count}")


資料健檢報告

✓ 總筆數: 30,000
✓ 欄位數: 23
✓ 欄位: ['device_id', 'device_mode_name', 'order_created_at', 'order_id', 'file_count', 'loc_1', 'loc_2', 'system_name', 'queue_duration_seconds', 'per_file_duration_avg_seconds', 'per_file_duration_max_seconds', 'per_file_duration_p95_seconds', 'device_duration_avg_seconds', 'device_duration_max_seconds', 'device_duration_p95_seconds', 'db_duration_avg_seconds', 'db_duration_max_seconds', 'db_duration_p95_seconds', 'inner_processing_duration_avg_seconds', 'inner_processing_duration_max_seconds', 'inner_processing_duration_p95_seconds', 'total_file_duration_minutes', 'total_duration_seconds']

⚠️  有 null 的欄位:
   device_mode_name: 5,675.0 (18.9%)
   loc_2: 6,717.0 (22.4%)

✓ order_id 重複: 0
✓ Unique devices: 200
✓ 時間範圍: 2026-03-01 03:52:29 ~ 2026-03-31 18:31:20
  天數: 30
✓ total_duration_seconds <= 0: 0
✓ file_count <= 0: 0


In [4]:
# 0. Schema 驗證
with open('../config/schema.yaml') as f:
    schema = yaml.safe_load(f)
expected_cols = [field['name'] for field in schema['fields']]
actual_cols = list(df.columns)
missing = set(expected_cols) - set(actual_cols)
extra = set(actual_cols) - set(expected_cols)

print("=" * 60)
print("Schema 驗證")
print("=" * 60)
if missing:
    print(f"\n✗ 缺少欄位: {sorted(missing)}")
else:
    print(f"\n✓ 所有 schema 欄位都存在 ({len(expected_cols)} 個)")
if extra:
    print(f"ℹ️  額外欄位（不在 schema 中）: {sorted(extra)}")

# Check nullable
for field in schema['fields']:
    col = field['name']
    if col in df.columns and not field.get('nullable', False):
        n = df[col].isnull().sum()
        if n > 0:
            print(f"✗ {col}: 有 {n} 筆 null，但 schema 定義為 non-nullable")

Schema 驗證

✓ 所有 schema 欄位都存在 (23 個)


In [5]:
# 7. 數值欄位 describe
print("\n=== 數值欄位統計 ===")
print(df.describe().round(1).to_string())



=== 數值欄位統計 ===
                    order_created_at  file_count  queue_duration_seconds  per_file_duration_avg_seconds  per_file_duration_max_seconds  per_file_duration_p95_seconds  device_duration_avg_seconds  device_duration_max_seconds  device_duration_p95_seconds  db_duration_avg_seconds  db_duration_max_seconds  db_duration_p95_seconds  inner_processing_duration_avg_seconds  inner_processing_duration_max_seconds  inner_processing_duration_p95_seconds  total_file_duration_minutes  total_duration_seconds
count                          30000     30000.0                 30000.0                        30000.0                        30000.0                        30000.0                      30000.0                      30000.0                      30000.0                  30000.0                  30000.0                  30000.0                                30000.0                                30000.0                                30000.0                      30000.0             

In [6]:
# 8. 關鍵 percentiles（記下來，後續步驟會用到）
print("\n=== 關鍵 Percentiles（供後續步驟參考）===")
for col in ['file_count', 'queue_duration_seconds', 'device_duration_avg_seconds',
            'db_duration_avg_seconds', 'total_duration_seconds']:
    pcts = df[col].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(1)
    print(f"\n{col}:")
    print(f"  P25={pcts[0.25]}, P50={pcts[0.5]}, P75={pcts[0.75]}, "
          f"P90={pcts[0.9]}, P95={pcts[0.95]}, P99={pcts[0.99]}, max={df[col].max()}")



=== 關鍵 Percentiles（供後續步驟參考）===

file_count:
  P25=43.0, P50=215.0, P75=714.2, P90=1441.0, P95=1897.0, P99=4256.0, max=4996

queue_duration_seconds:
  P25=4.0, P50=10.0, P75=22.0, P90=40.0, P95=62.0, P99=287.0, max=3631

device_duration_avg_seconds:
  P25=2.0, P50=3.0, P75=4.0, P90=5.0, P95=6.0, P99=20.0, max=599

db_duration_avg_seconds:
  P25=1.0, P50=2.0, P75=2.0, P90=2.0, P95=2.0, P99=3.0, max=150

total_duration_seconds:
  P25=79.0, P50=267.0, P75=961.0, P90=2018.0, P95=3062.0, P99=6484.0, max=29911


In [7]:
# 9. 前 5 筆
print("\n=== 前 5 筆資料 ===")
print(df.head().to_string())



=== 前 5 筆資料 ===
  device_id device_mode_name    order_created_at    order_id  file_count  loc_1   loc_2 system_name  queue_duration_seconds  per_file_duration_avg_seconds  per_file_duration_max_seconds  per_file_duration_p95_seconds  device_duration_avg_seconds  device_duration_max_seconds  device_duration_p95_seconds  db_duration_avg_seconds  db_duration_max_seconds  db_duration_p95_seconds  inner_processing_duration_avg_seconds  inner_processing_duration_max_seconds  inner_processing_duration_p95_seconds  total_file_duration_minutes  total_duration_seconds
0  DEV-0124              NaN 2026-03-03 23:32:42  ORD-000000          21  FAB-C  AREA-2    SYS-BETA                      23                              4                             19                             13                            1                           12                            6                        2                        5                        4                                      1                 